In [1]:
import pandas as pd
import os

# === Define base folder and file paths ===
base_folder = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1"
config_file = os.path.join(base_folder, "List_of_Config.csv")
step4_file = os.path.join(base_folder, "step4_detected_yml_files.csv")
output_file = os.path.join(base_folder, "yml_file_comparison.csv")

# === Load CSVs ===
df_config = pd.read_csv(config_file)
df_step4 = pd.read_csv(step4_file)

# === Filter only .yml or .yaml files ===
df_config = df_config[df_config['file_path'].str.endswith(('.yml', '.yaml'), na=False)]
df_step4 = df_step4[df_step4['file_path'].str.endswith(('.yml', '.yaml'), na=False)]

# === Create a unique key to compare ===
df_config['yml_identifier'] = df_config['html_url'] + ' | ' + df_config['file_path']
df_step4['yml_identifier'] = df_step4['html_url'] + ' | ' + df_step4['file_path']

# === Perform full outer merge with indicator ===
df_merged = pd.merge(
    df_config[['yml_identifier']], 
    df_step4[['yml_identifier']], 
    on='yml_identifier', 
    how='outer', 
    indicator=True
)

# === Classify source of each yml ===
df_merged['source'] = df_merged['_merge'].map({
    'both': 'exists_in_both',
    'left_only': 'only_in_List_of_Config',
    'right_only': 'only_in_step4'
})
df_merged.drop(columns=['_merge'], inplace=True)

# === Split identifier for clarity ===
df_merged[['html_url', 'file_path']] = df_merged['yml_identifier'].str.split(' \| ', expand=True)
df_merged = df_merged[['html_url', 'file_path', 'source']]

# === Save result ===
df_merged.to_csv(output_file, index=False)
print(f"Comparison saved to: {output_file}")


Comparison saved to: C:\Android Mobile App\Step2_Clone_Repo\Type_1\yml_file_comparison.csv
